# A3 — reproduction and claim verification (executable)

A3 is about *execution evidence*, so this notebook re-checks the claims the A3 documents make, against two
things that are committed here: the **frozen third-party checkout** and the **raw logs** of the runs.

It does not re-run `npm install` / `npm run build` (minutes, and the logs are the artefact of record) — it
verifies that the artefacts say what the documents claim, and that the code facts (line numbers, sizes,
dependencies, the predicate) still hold.

How to run: `python3 -m nbconvert --to notebook --execute --inplace notebooks/a3-reproduction-verification.ipynb`

In [1]:
import json, os, re, subprocess, sys

root = os.getcwd()
while not os.path.exists(os.path.join(root, "doc", "assignments", "a3")):
    parent = os.path.dirname(root); assert parent != root, "repository root not found"; root = parent
A3 = os.path.join(root, "doc/assignments/a3")
# the A3 working tree sits next to the repository folder (see doc/assignments/a3/reproduction-guide.md)
CHECKOUT = os.path.join(os.path.dirname(root), "A3", "work", "UCAS-course-planner")
print("repository root     :", os.path.basename(root))
print("A3 evidence         :", os.path.relpath(A3, root))
print("frozen checkout     :", "present" if os.path.isdir(CHECKOUT) else "MISSING (see reproduction-guide.md)")

repository root     : project
A3 evidence         : doc/assignments/a3
frozen checkout     : present


## 1. The identity A3 froze

In [2]:
pkg = json.load(open(os.path.join(CHECKOUT, "package.json"), encoding="utf-8"))
sha = subprocess.run(["git", "-C", CHECKOUT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit            :", sha)
print("expected          : 68e7080503084a41f42db6e0e9ff94c0e9ef2f19", "->", sha == "68e7080503084a41f42db6e0e9ff94c0e9ef2f19")
print("name/version      :", pkg["name"], pkg["version"])
print("engines.node      :", pkg["engines"]["node"])
print("scripts           :", sorted(pkg["scripts"]))
print("has a test script :", "test" in pkg["scripts"], "(documented gap if False)")
print("vinext            :", pkg["dependencies"].get("vinext"), "| vite:", pkg["devDependencies"].get("vite"))

commit            : 68e7080503084a41f42db6e0e9ff94c0e9ef2f19
expected          : 68e7080503084a41f42db6e0e9ff94c0e9ef2f19 -> True
name/version      : ucas-graduate-course-planner 0.1.0
engines.node      : >=22.13.0
scripts           : ['build', 'dev', 'format', 'lint', 'start']
has a test script : False (documented gap if False)
vinext            : 1.0.0-beta.5 | vite: 8.0.13


## 2. The code facts the architecture map cites

In [3]:
page = open(os.path.join(CHECKOUT, "app/page.tsx"), encoding="utf-8").read()
lines = page.splitlines()
def at(n): return lines[n-1].strip()
for n, expect in [(112, "PLAN_STORAGE_KEY"), (447, "coursesConflict"), (464, "conflictsWithPlan"),
                  (563, "fetch("), (578, "storageVersion"), (621, "setItem")]:
    ok = expect in at(n)
    print("L%-4d %-18s %s  | %s" % (n, expect, "OK" if ok else "MISMATCH", at(n)[:78]))
print()
print("page.tsx lines     :", len(lines), "(architecture map says 1879)")
print("components/ui files:", len(os.listdir(os.path.join(CHECKOUT, "components/ui"))), "(map says 60)")
print("app components     :", [d for d in os.listdir(os.path.join(CHECKOUT, "components")) if d != "ui"], "(map says none)")

L112  PLAN_STORAGE_KEY   OK  | const PLAN_STORAGE_KEY = 'ucas-graduate-course-planner-v3';
L447  coursesConflict    OK  | function coursesConflict(left: Course, right: Course) {
L464  conflictsWithPlan  OK  | function conflictsWithPlan(course: Course, plan: Course[]) {
L563  fetch(             OK  | fetch('/data/courses.json')
L578  storageVersion     OK  | if (parsed.storageVersion === 1) {
L621  setItem            OK  | window.localStorage.setItem(

page.tsx lines     : 1879 (architecture map says 1879)
components/ui files: 60 (map says 60)
app components     : [] (map says none)


## 3. The predicate — the same three conditions as A2's decision table

A2 says a clash requires *same weekday ∧ overlapping periods ∧ overlapping weeks*. The claim in A3 is that the
third-party code implements the same rule. The code is quoted rather than paraphrased:

In [4]:
body = page[page.index("function coursesConflict"):page.index("function conflictsWithPlan")]
print(body)
conds = {"same day": "leftSession.day === rightSession.day" in body,
         "weeks intersect": "parseWeeks" in body and "intersects" in body,
         "periods intersect": "parsePeriods" in body and "intersects" in body}
print("conditions present:", conds)
print("matches A2's three-condition rule:", all(conds.values()))
guard = page[page.index("function conflictsWithPlan"):page.index("function conflictsWithPlan") + 260]
print("self-comparison excluded:", "code !== course.code" in guard)

function coursesConflict(left: Course, right: Course) {
  return left.sessions.some((leftSession) =>
    right.sessions.some(
      (rightSession) =>
        leftSession.day === rightSession.day &&
        intersects(
          parseWeeks(String(leftSession.weeks)),
          parseWeeks(String(rightSession.weeks)),
        ) &&
        intersects(
          parsePeriods(String(leftSession.periods)),
          parsePeriods(String(rightSession.periods)),
        ),
    ),
  );
}


conditions present: {'same day': True, 'weeks intersect': True, 'periods intersect': True}
matches A2's three-condition rule: True
self-comparison excluded: True


## 4. The end-to-end evidence — the logs must say what the documents claim

In [5]:
log = open(os.path.join(A3, "evidence/logs/08-e2e.log"), encoding="utf-8").read()
checks = {
 "positive badge 3 处时间冲突": "3 处时间冲突" in log,
 "positive labels = 3": "conflictCellLabels=3" in log,
 "control badge 无时间冲突": "无时间冲突" in log,
 "control labels = 0": "conflictCellLabels=0" in log,
}
for k, v in checks.items():
    print("  %-28s %s" % (k, "OK" if v else "MISSING"))
print()
for name in ("04-npm-ci.log", "05-build.log", "06-lint.log", "07-start.log"):
    t = open(os.path.join(A3, "evidence/logs", name), encoding="utf-8").read()
    marker = {"04-npm-ci.log": "added 560 packages", "05-build.log": "Build complete",
              "06-lint.log": "0 warnings and 0 errors", "07-start.log": "port 3000"}[name]
    print("  %-16s contains %-24r %s" % (name, marker, "OK" if marker in t else "MISSING"))

  positive badge 3 处时间冲突       OK
  positive labels = 3          OK
  control badge 无时间冲突          OK
  control labels = 0           OK

  04-npm-ci.log    contains 'added 560 packages'     OK
  05-build.log     contains 'Build complete'         OK
  06-lint.log      contains '0 warnings and 0 errors' OK
  07-start.log     contains 'port 3000'              OK


## 5. The environment trap, as recorded

In [6]:
first = open(os.path.join(A3, "evidence/logs/01-npm-install.log"), encoding="utf-8").read()
failed = open(os.path.join(A3, "evidence/logs/02-build.log"), encoding="utf-8").read()
print("first install added (dev deps skipped):", "added 513 packages" in first)
print("first build failed on the missing plugin:", "Cannot find package '@openai/sites-vite-plugin'" in failed)
print("first build exit code recorded as 1     :", "EXIT=1" in failed)
print()
print("=> the guide's claim (environment, not the repository, caused the first failure) is consistent with these logs")

first install added (dev deps skipped): True
first build failed on the missing plugin: True
first build exit code recorded as 1     : True

=> the guide's claim (environment, not the repository, caused the first failure) is consistent with these logs


## 6. What this notebook does not establish

- it does not rebuild the JavaScript; the build logs are the artefact of record;
- it does not re-run the browser end-to-end; it checks that the log of that run says what the documents claim;
- the interactive path (search → add → re-render) remains *read*, not exercised.